# FPL Squad Optimizer

This Jupyter Notebook uses a knapsack algorithm to create an Fantasy Premier League squad of 15 players optimized around a specified metric (goals, clean sheets, points etc.)

## Import Libraries

In [22]:
# Standard libraries
import requests
import pandas as pd
import numpy as np
import copy
import sys

# libraries for postgres connection
import psycopg2
from sqlalchemy import create_engine
import pandas.io.sql as sqlio

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

# for env variables
import os
from dotenv import load_dotenv, get_key
load_dotenv()

# save env variables
SUPABASE_USER = get_key('.env', 'SUPABASE_USER')
SUPABASE_HOST = get_key('.env', 'SUPABASE_HOST')
SUPABASE_PASSWORD = get_key('.env', 'SUPABASE_PASSWORD')
SUPABASE_PORT = get_key('.env', 'SUPABASE_PORT')
SUPABASE_DB = get_key('.env', 'SUPABASE_DB')

# interactive shell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

True

## Fetch data from the FPL API and clean it

In [2]:
# FPL API for fixtures
url = 'https://fantasy.premierleague.com/api/fixtures/'
response = requests.get(url)
fixtures_json = response.json()

# store in pandas DF
fixtures_df = pd.DataFrame(fixtures_json)

In [3]:
# FPL API URL
url = 'https://fantasy.premierleague.com/api/bootstrap-static/'
response = requests.get(url)
json = response.json()

# JSON keys
json.keys()

dict_keys(['chips', 'events', 'game_settings', 'game_config', 'phases', 'teams', 'total_players', 'element_stats', 'element_types', 'elements'])

In [4]:
# storing json outputs as dataframes

elements_df = pd.DataFrame(json['elements'])
elements_types_df = pd.DataFrame(json['element_types'])
teams_df = pd.DataFrame(json['teams'])

In [5]:
# Pulling in player position into slim_elements_df

elements_df['position'] = elements_df.element_type.map(elements_types_df.set_index('id').singular_name)
elements_df['team_name'] = elements_df.team.map(teams_df.set_index('id').name)
elements_df['team_short_name'] = elements_df.team.map(teams_df.set_index('id').short_name)

In [6]:
# Filtering out only the necessary columns

slim_elements_df = elements_df[['id', 'first_name','second_name','web_name','team_name','team_short_name','position','news','selected_by_percent','in_dreamteam',
                                'now_cost','form','points_per_game','minutes','goals_scored','assists','clean_sheets','goals_conceded','yellow_cards','red_cards','saves','bonus',
                                'transfers_in','starts','value_season','total_points','influence','creativity','threat','ict_index']]

# numeric columns:

numeric_cols = ['selected_by_percent','form','points_per_game','value_season','influence','creativity','threat','ict_index']

In [7]:
# convering columns into numeric data type

for col in numeric_cols:
    slim_elements_df[col] = pd.to_numeric(slim_elements_df[col])


In [8]:
def convert_filename(string):
    return string.lower().replace(" ", "_")

# rename the web_name column
slim_elements_df.rename(columns = {'web_name':'name'}, inplace = True)

# actual cost of the player is now_cost/10
slim_elements_df['actual_cost'] = slim_elements_df['now_cost']/10

# creating additional metrics
slim_elements_df['games_completed'] = slim_elements_df['minutes']/90
slim_elements_df['points_per_90_mins'] = slim_elements_df['total_points']/slim_elements_df['games_completed']
slim_elements_df['ga_per_90_mins'] = (slim_elements_df['goals_scored']+slim_elements_df['assists'])/slim_elements_df['games_completed']
slim_elements_df['points_per_million'] = slim_elements_df['total_points']/slim_elements_df['actual_cost']
slim_elements_df['image_path'] = '/' + slim_elements_df['team_name'].apply(convert_filename) + '.svg'

# eligible players
eligible_players = slim_elements_df[slim_elements_df['news'] == '']

# create a dataframe with only differentials: owned by less than 20%
differentials = slim_elements_df.loc[(slim_elements_df['news'] == '') & (slim_elements_df['selected_by_percent'] <= 20)]

In [9]:
# create percentile columns for specific metrics
percentile_metrics = ['bonus', 'form', 'ict_index', 'points_per_game', 'points_per_million', 'total_points', 'goals_scored', 'assists', 'clean_sheets', 'ga_per_90_mins']

for metric in percentile_metrics:
    slim_elements_df[metric + '_percentile'] = slim_elements_df.groupby('position')[metric].rank(pct=True)

slim_elements_df.sort_values(by = 'total_points', ascending = False).head(10)

,id,first_name,second_name,name,team_name,team_short_name,position,news,selected_by_percent,in_dreamteam,...,bonus_percentile,form_percentile,ict_index_percentile,points_per_game_percentile,points_per_million_percentile,total_points_percentile,goals_scored_percentile,assists_percentile,clean_sheets_percentile,ga_per_90_mins_percentile
127,115,Maxim,De Cuyper,De Cuyper,Brighton,BHA,Defender,,33.1,True,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.993088,0.995392,0.993088,0.974026
136,124,Pascal,Groß,Groß,Brighton,BHA,Midfielder,,37.4,True,...,1.000000,1.000000,0.993289,0.993289,1.000000,1.000000,0.983221,1.000000,0.994966,0.956098
272,229,James,Tarkowski,Tarkowski,Everton,EVE,Defender,,18.3,True,...,0.995392,0.995392,0.898618,0.995392,0.986175,0.995392,0.935484,0.926267,0.951613,0.892857
184,165,João Pedro,Junqueira de Jesus,João Pedro,Chelsea,CHE,Forward,,68.0,True,...,0.993671,1.000000,0.974684,1.000000,1.000000,0.993671,0.987342,0.987342,0.386076,0.960784
491,411,Erling,Haaland,Haaland,Man City,MCI,Forward,,74.7,True,...,0.993671,0.981013,1.000000,0.987342,0.860759,0.993671,1.000000,0.411392,0.962025,0.843137
482,399,Rayan,Cherki,Cherki,Man City,MCI,Midfielder,,29.2,True,...,0.991611,0.971477,0.989933,0.989933,0.979866,0.996644,0.998322,0.946309,0.882550,0.960976
403,330,Jayden,Bogle,Bogle,Leeds,LEE,Defender,,8.5,True,...,0.986175,0.986175,0.981567,0.990783,0.995392,0.990783,0.993088,0.442396,0.993088,0.909091
474,391,Joško,Gvardiol,Gvardiol,Man City,MCI,Defender,,27.7,True,...,0.986175,0.976959,0.958525,0.981567,0.976959,0.986175,0.935484,1.000000,0.877880,0.980519
105,94,Kevin,Schade,Schade,Brentford,BRE,Midfielder,Shoulder injury - 75% chance of playing,15.7,True,...,0.981544,0.996644,0.953020,0.986577,0.993289,0.993289,0.983221,0.946309,0.882550,0.897561
151,504,Luka,Vušković,Vuskovic,Brighton,BHA,Defender,,4.3,True,...,0.900922,0.990783,0.928571,0.976959,0.990783,0.981567,0.935484,0.442396,0.993088,0.785714


## Python functions for the knapsack algorithm

In [10]:
def knapsack_solution(players, player_costs, player_values, max_cost, count):
    
    """
    function that returns the knapsack cost matrix.

    cost_matrix[i][j][k] = best total value using exactly k players
    from players[0..i] with total cost at most j.

    Assumes player_values are strictly positive for every player (see
    adjust_player_values), so 0 always means "infeasible".
    """
  
    num_players = len(players)
  
    cost_matrix = [[[0 for k in range(count+1)] for j in range(max_cost+1)] for i in range(num_players)]
    
    for i in range(num_players):
        for j in range(max_cost+1):
            for k in range(count+1):
                if k == 0:
                    cost_matrix[i][j][k] = 0
                    continue

                # Skip player i
                if i == 0:
                    best = 0
                else:
                    best = cost_matrix[i-1][j][k]

                # Take player i only from a feasible (k-1)-player solution
                # (for k == 1, the empty prior solution is always feasible)
                if player_costs[i] <= j:
                    if k == 1:
                        best = max(best, player_values[i])
                    elif i > 0 and cost_matrix[i-1][j - player_costs[i]][k - 1] > 0:
                        best = max(
                            best,
                            player_values[i] + cost_matrix[i-1][j - player_costs[i]][k - 1]
                        )

                cost_matrix[i][j][k] = best

    return cost_matrix
    

In [11]:
def get_used_items(players, player_costs, player_values, max_cost, count, cost_matrix):
    
    """
    function that returns the used players from the cost matrix.

    Reconstructs an exactly-`count` selection. When take/skip values are tied
    (e.g. zero-metric players), prefer taking so required slots are filled.
    """
    
    playerIndex = len(players) - 1
    currentCount = count
    marked = [0 for _ in range(len(players))]

    bestValue = None
    currentCost = 0
    
    for j in range(max_cost + 1):
        value = cost_matrix[playerIndex][j][count]
        if (bestValue is None) or (value > bestValue):
            currentCost = j
            bestValue = value

    # If every player has metric 0, every budget looks equally optimal at j=0.
    # Use the full budget so reconstruction can still pick `count` players.
    if bestValue == 0 and count > 0:
        currentCost = max_cost
    
    while playerIndex >= 0 and currentCount > 0:
        if playerIndex == 0:
            if (
                currentCount == 1
                and player_costs[0] <= currentCost
                and (
                    cost_matrix[0][currentCost][1] == player_values[0]
                    or player_values[0] == 0
                )
            ):
                marked[0] = 1
            break

        current_val = cost_matrix[playerIndex][currentCost][currentCount]
        can_take = player_costs[playerIndex] <= currentCost

        if can_take:
            take_val = (
                player_values[playerIndex]
                + cost_matrix[playerIndex - 1][currentCost - player_costs[playerIndex]][currentCount - 1]
            )
            # Prefer taking on ties so zero-value players still fill the squad
            if current_val == take_val:
                marked[playerIndex] = 1
                currentCost -= player_costs[playerIndex]
                currentCount -= 1

        playerIndex -= 1

    return marked
    

## Python functions to optimize keepers, defenders, midfielders, forwards

The knapsack is run separately per position and always selects **exactly** the FPL required counts: 2 goalkeepers, 5 defenders, 5 midfielders and 3 forwards (15 players total).

Steps:
1. Get every combination of 4 numbers adding up to 100 (budget split across GK / DEF / MID / FWD)
2. For each combination, run the knapsack for each position with a hard exact-count constraint
3. Discard any incomplete squads, then choose the combination with the highest target metric

In [12]:
REQUIRED_POSITION_COUNTS = {
    'Goalkeeper': 2,
    'Defender': 5,
    'Midfielder': 5,
    'Forward': 3,
}


def adjust_player_values(player_values):
    """
    Scale the metric and add +1 per player so the knapsack always prefers
    filling the exact required count, even when some metric values are 0.
    """
    return [float(v) * 1_000_000 + 1 for v in player_values]


def optimum_by_position(eligible_players, maximum_cost, opt_metric, position, count):
    """Select exactly `count` players for a position within `maximum_cost` (in £m)."""

    max_cost = int(maximum_cost * 10)

    pos_df = eligible_players[eligible_players['position'] == position].reset_index(drop=True)
    players = pos_df.index.tolist()
    player_costs = pos_df['now_cost'].astype(int).tolist()
    player_values = adjust_player_values(pos_df[opt_metric].fillna(0).tolist())

    empty_cols = [
        'first_name', 'second_name', 'name', 'team_name', 'team_short_name',
        'position', 'selected_by_percent', 'actual_cost', 'total_points', opt_metric
    ]

    if len(players) < count:
        return pd.DataFrame(columns=empty_cols)

    # Impossible to afford `count` players within this budget
    cheapest_total = sum(sorted(player_costs)[:count])
    if cheapest_total > max_cost:
        return pd.DataFrame(columns=empty_cols)

    cost_matrix = knapsack_solution(players, player_costs, player_values, max_cost, count)
    used_players = get_used_items(players, player_costs, player_values, max_cost, count, cost_matrix)

    player_indices = [i for i, used in enumerate(used_players) if used == 1]

    if len(player_indices) != count:
        return pd.DataFrame(columns=empty_cols)

    players_df = pos_df.iloc[player_indices]
    final = players_df[empty_cols]

    return final.loc[:, ~final.columns.duplicated()].copy()


def optimum_keepers(eligible_players, maximum_cost, opt_metric):
    return optimum_by_position(eligible_players, maximum_cost, opt_metric, 'Goalkeeper', 2)

In [13]:
def optimum_defence(eligible_players, maximum_cost, opt_metric):
    return optimum_by_position(eligible_players, maximum_cost, opt_metric, 'Defender', 5)

In [14]:
def optimum_midfield(eligible_players, maximum_cost, opt_metric):
    return optimum_by_position(eligible_players, maximum_cost, opt_metric, 'Midfielder', 5)

In [15]:
def optimum_attack(eligible_players, maximum_cost, opt_metric):
    return optimum_by_position(eligible_players, maximum_cost, opt_metric, 'Forward', 3)

# Cost Breakdowns

The following functions provide all the combination sums that add up to 100.

To avoid longer runtimes for the optimization functions, the costs for each section has a minimum threshold:
1. Keepers: 8 MM
2. Defence: 25 MM
3. Midfield: 30 MM
4. Attack: 20 MM

In [16]:
# Functions to get all the sum combinations

def print_all_sum_rec(target, current_sum, start, output, result):
    if current_sum == target:
        output.append(copy.copy(result))

    for i in range(start, target):
        temp_sum = current_sum + i
        if temp_sum <= target:
            result.append(i)
            print_all_sum_rec(target, temp_sum, i, output, result)
            result.pop()
        else:
            return

def print_all_sum(target):
    output = []
    result = []
    print_all_sum_rec(target, 0, 4, output, result)
    return output


In [17]:
# Function that selects only the combinations with 4 numbers

def cost_breakdown(number):
    breakdown = print_all_sum(number)
    combinations = []
    for i in breakdown:
        if len(i) == 4:
            if (i[0] >= 8) and (i[1] >= 25) and (i[2] >= 30) and (i[3] >= 20):
                combinations.append(i)
    return combinations

In [18]:
# Function that returns the best cost breakdown (keepers - defence - midfield - attack) for the chosen metric

def best_cost_breakdown(opt_metric):
    costs_combinations = cost_breakdown(100)

    comb_df = pd.DataFrame(columns = ['costs', 'total_cost', opt_metric, 'squad_size'])
    
    for costs in costs_combinations:
        
        gk = optimum_keepers(eligible_players, costs[0], opt_metric)
        dfnc = optimum_defence(eligible_players, costs[1], opt_metric)
        mid = optimum_midfield(eligible_players, costs[2], opt_metric)
        att = optimum_attack(eligible_players, costs[3], opt_metric)

        # Only accept full FPL squads: 2 GK, 5 DEF, 5 MID, 3 FWD
        if len(gk) != 2 or len(dfnc) != 5 or len(mid) != 5 or len(att) != 3:
            continue
        
        final = pd.concat([gk, dfnc, mid, att])
        total_cost = final['actual_cost'].sum()
        optimized_metric = final[opt_metric].sum()
        cost_details = [costs, total_cost, optimized_metric, len(final)]
        
        comb_df.loc[len(comb_df)] = cost_details

    if comb_df.empty:
        raise ValueError(
            f"No valid 15-player squad (2 GK / 5 DEF / 5 MID / 3 FWD) found for metric '{opt_metric}'."
        )

    comb_df[opt_metric] = pd.to_numeric(comb_df[opt_metric])

    return comb_df.sort_values(by=[opt_metric], ascending=False).reset_index(drop=True).head(1)

## Final Optimization Function

A final function that takes in the metric and returns an optimized squad.

In [19]:
def squad_optimizer(opt_metric, eligible_players):
    
    costs = best_cost_breakdown(opt_metric)['costs'].iloc[0]
    
    keepers = optimum_keepers(eligible_players, costs[0], opt_metric)
    defence = optimum_defence(eligible_players, costs[1], opt_metric)
    midfield = optimum_midfield(eligible_players, costs[2], opt_metric)
    attack = optimum_attack(eligible_players, costs[3], opt_metric)

    final_squad_df = pd.concat([keepers, defence, midfield, attack]).reset_index(drop=True)

    position_counts = final_squad_df['position'].value_counts().to_dict()
    for position, required in REQUIRED_POSITION_COUNTS.items():
        actual = position_counts.get(position, 0)
        if actual != required:
            raise ValueError(
                f"Expected {required} {position}(s), got {actual}. "
                f"Squad size: {len(final_squad_df)}."
            )

    if len(final_squad_df) != 15:
        raise ValueError(f"Expected 15 players, got {len(final_squad_df)}.")

    return final_squad_df

In [20]:
squad = squad_optimizer('total_points', eligible_players)

In [21]:
squad

,first_name,second_name,name,team_name,team_short_name,position,selected_by_percent,actual_cost,total_points
0,David,Raya Martín,Raya,Arsenal,ARS,Goalkeeper,43.7,6.1,33
1,Konstantinos,Tzolakis,Tzolakis,Hull City,HUL,Goalkeeper,13.4,4.7,36
2,Maxim,De Cuyper,De Cuyper,Brighton,BHA,Defender,33.1,5.0,53
3,Luka,Vušković,Vuskovic,Brighton,BHA,Defender,4.3,5.0,40
4,James,Tarkowski,Tarkowski,Everton,EVE,Defender,18.3,6.2,46
5,Jayden,Bogle,Bogle,Leeds,LEE,Defender,8.5,4.6,43
6,Joško,Gvardiol,Gvardiol,Man City,MCI,Defender,27.7,5.8,41
7,Vitaly,Janelt,Janelt,Brentford,BRE,Midfielder,2.5,5.0,31
8,Pascal,Groß,Groß,Brighton,BHA,Midfielder,37.4,6.0,53
9,Kiernan,Dewsbury-Hall,Dewsbury-Hall,Everton,EVE,Midfielder,5.5,6.6,33


# Write DataFrame to cloud PostgreSQL DB (Supabase) 

In [23]:
# establish connection

conn = psycopg2.connect(
    database=SUPABASE_DB, 
    user=SUPABASE_USER, 
    password=SUPABASE_PASSWORD, 
    host=SUPABASE_HOST, 
    port=SUPABASE_PORT,
    sslmode='require'
)

In [24]:
# Setting auto commit true
conn.autocommit = True

# Creating a cursor object using the cursor() method
cursor = conn.cursor()


In [25]:
# List of metrics to create tables for 
optimizing_metrics = ['points_per_game','total_points','ict_index','points_per_million']

# Delete already existing tables for these metrics
for metric in optimizing_metrics:

    table_name = 'public.optimal_squad_' + metric
    drop_query = 'DROP TABLE IF EXISTS ' + table_name
    cursor.execute(drop_query)

In [26]:
# Create SQL alchemy engine
engine_url = 'postgresql://' + SUPABASE_USER + ':' + SUPABASE_PASSWORD + '@' + SUPABASE_HOST + '/' + SUPABASE_DB
engine = create_engine(engine_url)

# Load tables in supabase
for metric in optimizing_metrics:
    print(metric)
    squad = squad_optimizer(metric, eligible_players)
    squad = pd.merge(squad, slim_elements_df[['first_name', 'second_name', 'id', 'image_path']], on=['first_name', 'second_name'], how='left')
    squad.to_sql('optimal_squad_' + metric, engine, schema='public', index=False)


points_per_game


15

total_points


15

ict_index


15

points_per_million


15

# Load cleaned raw data from API into Supabase

The raw cleaned data from the API can be used for other ad-hoc analyses

In [47]:
# Delete the contents of the table and load the dataframe
cursor.execute('''DROP TABLE IF EXISTS public.dim_fpl_players''')

In [48]:
# Use sqlalchemy engine to write to the DB
engine_url = 'postgresql://' + SUPABASE_USER + ':' + SUPABASE_PASSWORD + '@' + SUPABASE_HOST + '/' + SUPABASE_DB

engine = create_engine(engine_url)
slim_elements_df.to_sql('dim_fpl_players', engine, schema='public', index=False)

49

In [49]:
# close cursor and connection

cursor.close()
conn.close()